# Notebook 02 - Cleaning & Preprocessing: Vietnam Student Dataset

Notebook này thực hiện quá trình làm sạch và tiền xử lý bộ dữ liệu khảo sát sinh viên Việt Nam nhằm tạo ra bộ dữ liệu sạch, nhất quán và phù hợp cho bước Exploratory Data Analysis (EDA) trong Notebook 03.

Notebook được xây dựng dựa trên cùng logic với Notebook 02 của bộ dữ liệu quốc tế nhằm đảm bảo tính nhất quán trong toàn bộ pipeline nghiên cứu Digital Burnout.


# 0. Set Up
Thiết lập môi trường cần thiết cho quá trình Cleaning & Preprocessing.
Phần này thực hiện:

- Import thư viện.
- Thiết lập cấu hình hiển thị.
- Khai báo đường dẫn input và output.

In [1]:
# Import các thư viện phục vụ xử lý dữ liệu

import pandas as pd
import numpy as np

import os


# Thiết lập cấu hình hiển thị dữ liệu

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    100
)

In [2]:
# Thiết lập đường dẫn dữ liệu đầu vào và đầu ra

input_path = os.path.join(
    "..",
    "..",
    "data",
    "raw",
    "vietnam_dataset",
    "vn_digital_burnout.csv"
)

print("Đã thiết lập đường dẫn dữ liệu.")

Đã thiết lập đường dẫn dữ liệu.


In [3]:
# Kiểm tra sự tồn tại của file dữ liệu đầu vào

if os.path.exists(input_path):
    print("Đã tìm thấy bộ dữ liệu đầu vào.")
else:
    print("Không tìm thấy bộ dữ liệu đầu vào.")

Đã tìm thấy bộ dữ liệu đầu vào.


# 1. Load Dataset

Đọc dữ liệu khảo sát sinh viên Việt Nam và kiểm tra trạng thái ban đầu trước khi thực hiện cleaning.

Phần này thực hiện:

- Load file CSV.
- Kiểm tra kích thước dataset.
- Kiểm tra kiểu dữ liệu ban đầu.

In [4]:
# Đọc bộ dữ liệu khảo sát sinh viên Việt Nam

raw_dataset = pd.read_csv(
    input_path
)

print("Đã tải dữ liệu thành công.")

Đã tải dữ liệu thành công.


In [5]:
# Kiểm tra kích thước dữ liệu ban đầu

raw_shape = raw_dataset.shape

print("Kích thước dữ liệu ban đầu:")
print(f"Số dòng: {raw_shape[0]}")
print(f"Số cột: {raw_shape[1]}")

Kích thước dữ liệu ban đầu:
Số dòng: 641
Số cột: 25


In [6]:
# Kiểm tra kiểu dữ liệu ban đầu

raw_data_types = raw_dataset.dtypes.to_frame()

raw_data_types.columns = [
    "data_type"
]


raw_data_types

,data_type
Giới tính của bạn?,object
Năm sinh?,object
Bạn đang ở giai đoạn nào?,object
Hình thức học tập / làm việc chính hiện tại:,object
Mô tả cách bạn dùng thiết bị số hiện tại:,object
Trung bình mỗi ngày bạn dùng điện thoại và máy tính tổng cộng bao nhiêu giờ (gộp cả học/làm lẫn giải trí)?,object
"Bạn dành bao nhiêu giờ mỗi ngày cho mạng xã hội (TikTok, Instagram, Facebook, YouTube Shorts...)?",object
"Bạn có thường xuyên cuộn feed (TikTok, Reels, Shorts...) liên tục không có mục đích rõ ràng? Nếu có, mỗi ngày khoảng bao lâu?",object
Bạn có thường dùng điện thoại hoặc máy tính sau 22 giờ đêm?,object
Bạn nhận khoảng bao nhiêu thông báo trên tất cả ứng dụng mỗi ngày?,object


# 2. Data Cleaning

Thực hiện các bước làm sạch dữ liệu nhằm chuyển đổi bộ dữ liệu khảo sát dạng raw thành dataset có cấu trúc nhất quán và phù hợp cho các bước phân tích tiếp theo.

Phần này bao gồm:

- Chuẩn hóa tên biến.
- Chuẩn hóa giá trị khảo sát.
- Xử lý missing values.
- Xử lý duplicate records.
- Kiểm tra dữ liệu không hợp lệ.
- Chuẩn hóa kiểu dữ liệu.

## 2.1 Column Standardization

Chuẩn hóa tên biến từ dạng câu hỏi khảo sát tiếng Việt sang định dạng biến chuẩn snake_case.

Các tên biến mới được xây dựng dựa trên:

- Ý nghĩa của câu hỏi khảo sát.
- Mapping với bộ dữ liệu quốc tế.
- Khả năng sử dụng trong các bước phân tích Machine Learning.

In [7]:
# Tạo bản sao dữ liệu trước khi làm sạch

cleaned_dataset = raw_dataset.copy()

In [8]:
# Chuẩn hóa tên biến từ câu hỏi khảo sát sang snake_case

column_mapping = {

    "Giới tính của bạn?":
        "gender",

    "Năm sinh?":
        "birth_year",

    "Bạn đang ở giai đoạn nào?":
        "education_stage",

    "Hình thức học tập / làm việc chính hiện tại:":
        "work_mode",

    "Mô tả cách bạn dùng thiết bị số hiện tại:":
        "device_usage_type",

    "Trung bình mỗi ngày bạn dùng điện thoại và máy tính tổng cộng bao nhiêu giờ  (gộp cả học/làm lẫn giải trí)?":
        "daily_screen_time",

    "Bạn dành bao nhiêu giờ mỗi ngày cho mạng xã hội (TikTok, Instagram, Facebook, YouTube Shorts...)?":
        "social_media_hours",

    "Bạn có thường xuyên cuộn feed (TikTok, Reels, Shorts...) liên tục không có mục đích rõ ràng? Nếu có, mỗi ngày khoảng bao lâu?":
        "doomscrolling_duration",

    "Bạn có thường dùng điện thoại hoặc máy tính sau 22 giờ đêm?":
        "late_night_device_usage",

    "Bạn nhận khoảng bao nhiêu thông báo trên tất cả ứng dụng mỗi ngày?":
        "notification_count",

    "Bạn mở khóa điện thoại khoảng bao nhiêu lần mỗi ngày?":
        "smartphone_unlocks",

    "Trong một ngày học/làm việc, bạn chuyển qua lại giữa ứng dụng/tab khoảng bao nhiêu lần?":
        "app_switch_frequency",

    "Bạn tự đánh giá khả năng tập trung của mình dạo này ở mức nào?":
        "concentration_score",

    "Trong 1 giờ học/làm việc, bạn bị kéo ra bởi điện thoại, thông báo hoặc tab khác bao nhiêu lần?":
        "distraction_frequency",

    "Trong một ngày, bạn có bao nhiêu lần ngồi học hoặc làm việc liên tục ít nhất 25 phút mà không mở điện thoại hay chuyển tab?":
        "focus_sessions",

    "Mỗi ngày bạn dành được bao nhiêu giờ thực sự tập trung sâu vào việc học/làm - không mạng xã hội, không thông báo, không multitask?":
        "deep_work_hours",

    "Nhìn lại hôm qua, bạn hoàn thành được khoảng bao nhiêu phần trăm những việc mình đã đặt ra (bài tập, deadline, công việc, nhiệm vụ cá nhân...)?":
        "task_completion_rate",

    "Trung bình bạn ngủ bao nhiêu tiếng mỗi đêm (không tính ngủ trưa)?":
        "sleep_hours",

    "Dạo này bạn thấy giấc ngủ của mình như thế nào? Khi thức dậy có thấy khỏe và tỉnh táo không?":
        "sleep_quality",

    "Mức độ có động lực để học tập hoặc làm việc của bạn dạo này ở mức nào?":
        "motivation_level",

    "Mình cảm thấy bã người, kiệt sức chỉ vì phải liên tục nhìn màn hình và xử lý thông tin cả ngày, kể cả khi làm những thứ mình thích.":
        "emotional_exhaustion",

    "Mình thấy căng thẳng, lo lắng khi thấy tin nhắn chưa đọc, thông báo chưa xử lý, hoặc deadline đang chồng chất.":
        "stress_level",

    "Mình cảm thấy đầu nặng, mệt mỏi và khó tắt não ngay cả khi đã tắt máy hay bỏ điện thoại xuống.":
        "mental_fatigue",

    "Con người cần hô hấp để sống đúng không?":
        "attention_check_question",

    "Bạn còn đang thực hiện khảo sát này đúng không? (Nếu đúng vui lòng chọn số 9)":
        "attention_check_score"
}


cleaned_dataset = cleaned_dataset.rename(
    columns=column_mapping
)


print("Đã chuẩn hóa tên biến.")

Đã chuẩn hóa tên biến.


In [9]:
# Kiểm tra danh sách biến sau khi chuẩn hóa

cleaned_dataset.columns.tolist()

['gender',
 'birth_year',
 'education_stage',
 'work_mode',
 'device_usage_type',
 'daily_screen_time',
 'social_media_hours',
 'doomscrolling_duration',
 'late_night_device_usage',
 'notification_count',
 'smartphone_unlocks',
 'attention_check_question',
 'app_switch_frequency',
 'concentration_score',
 'distraction_frequency',
 'focus_sessions',
 'deep_work_hours',
 'task_completion_rate',
 'sleep_hours',
 'sleep_quality',
 'attention_check_score',
 'motivation_level',
 'emotional_exhaustion',
 'stress_level',
 'mental_fatigue']

## 2.2 Attention Check Processing

Kiểm tra tính hợp lệ của phản hồi khảo sát thông qua các câu hỏi attention check.

Dataset chứa hai biến phục vụ kiểm tra chất lượng người trả lời:

- `attention_check_question`
- `attention_check_score`

Quy trình xử lý:

1. Kiểm tra giá trị attention check.
2. Loại bỏ các bản ghi không đạt yêu cầu.
3. Xóa các biến attention check khỏi dataset sau khi hoàn thành kiểm tra.

In [10]:
# Kiểm tra phân bố giá trị của attention check

attention_check_summary = cleaned_dataset[
    [
        "attention_check_question",
        "attention_check_score"
    ]
].value_counts()


attention_check_summary

attention_check_question  attention_check_score
Có                        9                        584
                          5                          6
                          6                          5
                          2                          5
                          8                          5
                          4                          4
                          10                         4
                          7                          4
Mình không biết           9                          4
Có                        3                          3
Không                     7                          2
Mình không biết           3                          2
Không                     9                          2
                          4                          2
                          10                         2
                          5                          1
                          2                          1
                 

In [11]:
# Lọc các phản hồi đạt yêu cầu attention check

before_attention_check = len(cleaned_dataset)


cleaned_dataset = cleaned_dataset[
    cleaned_dataset["attention_check_score"] == 9
]


after_attention_check = len(cleaned_dataset)


print(
    f"Số bản ghi trước kiểm tra: {before_attention_check}"
)

print(
    f"Số bản ghi sau kiểm tra: {after_attention_check}"
)

Số bản ghi trước kiểm tra: 641
Số bản ghi sau kiểm tra: 590


In [12]:
# Xóa các biến attention check sau khi hoàn thành kiểm tra

attention_check_columns = [
    "attention_check_question",
    "attention_check_score"
]


cleaned_dataset = cleaned_dataset.drop(
    columns=attention_check_columns
)


print("Đã loại bỏ các biến attention check.")

Đã loại bỏ các biến attention check.


## 2.5 Text Value Standardization

Chuẩn hóa các giá trị dạng text trước khi thực hiện encoding nhằm tránh sai lệch do khoảng trắng hoặc ký tự thừa.

In [13]:
# Chuẩn hóa khoảng trắng trong các biến dạng text

object_columns = (
    cleaned_dataset
    .select_dtypes(include="object")
    .columns
)


for column in object_columns:

    cleaned_dataset[column] = (
        cleaned_dataset[column]
        .str.strip()
    )


print("Đã chuẩn hóa giá trị text.")

Đã chuẩn hóa giá trị text.


# 2.3 Survey Response Encoding

Chuyển đổi các câu trả lời khảo sát dạng văn bản sang dạng số có cấu trúc nhằm chuẩn bị dữ liệu cho các bước phân tích tiếp theo.

Các biến khảo sát được phân loại thành:

1. Biến categorical:
    - Không có thứ tự tự nhiên.
    - Được giữ nguyên dạng text.

2. Biến ordinal:
    - Có thứ tự mức độ.
    - Được mã hóa bằng Ordinal Encoding.

3. Biến Likert Scale:
    - Đã có giá trị số.
    - Chỉ kiểm tra kiểu dữ liệu.

## 2.3.1 Digital Exposure Encoding

Mã hóa các biến phản ánh mức độ tiếp xúc với thiết bị số.

- daily_screen_time
- social_media_hours
- doomscrolling_duration
- late_night_device_usage
- notification_count
- smartphone_unlocks
- app_switch_frequency

In [14]:
# Mã hóa tổng thời gian sử dụng thiết bị số mỗi ngày

daily_screen_time_mapping = {

    "Dưới 4 giờ": 0,
    "4-6 giờ": 1,
    "6-8 giờ": 2,
    "8-10 giờ": 3,
    "Trên 10 giờ": 4

}

cleaned_dataset["daily_screen_time"] = (
    cleaned_dataset["daily_screen_time"]
    .replace(daily_screen_time_mapping)
)

print("Đã mã hóa daily_screen_time.")

Đã mã hóa daily_screen_time.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\3152699750.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(daily_screen_time_mapping)


In [15]:
# Mã hóa thời gian sử dụng mạng xã hội mỗi ngày

social_media_mapping = {

    "Dưới 1 giờ": 0,
    "1-2 giờ": 1,
    "2-4 giờ": 2,
    "Trên 4 giờ": 3

}

cleaned_dataset["social_media_hours"] = (
    cleaned_dataset["social_media_hours"]
    .replace(social_media_mapping)
)

print("Đã mã hóa social_media_hours.")

Đã mã hóa social_media_hours.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\3557182661.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(social_media_mapping)


In [16]:
# Mã hóa thời gian doomscrolling mỗi ngày

doomscrolling_mapping = {

    "Không bao giờ hoặc rất hiếm": 0,
    "Dưới 30 phút/ngày": 1,
    "30 phút - 1 giờ/ngày": 2,
    "1-2 giờ/ngày": 3,
    "Trên 2 giờ/ngày": 4

}

cleaned_dataset["doomscrolling_duration"] = (
    cleaned_dataset["doomscrolling_duration"]
    .replace(doomscrolling_mapping)
)

print("Đã mã hóa doomscrolling_duration.")

Đã mã hóa doomscrolling_duration.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\2591776933.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(doomscrolling_mapping)


In [17]:
# Mã hóa tần suất sử dụng thiết bị sau 22 giờ

late_night_mapping = {

    "Không bao giờ": 0,
    "1-2 lần/tuần": 1,
    "3-5 lần/tuần": 2,
    "Hầu như mỗi tối": 3

}

cleaned_dataset["late_night_device_usage"] = (
    cleaned_dataset["late_night_device_usage"]
    .replace(late_night_mapping)
)

print("Đã mã hóa late_night_device_usage.")

Đã mã hóa late_night_device_usage.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\326690723.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(late_night_mapping)


In [18]:
# Mã hóa số lượng thông báo nhận được mỗi ngày

notification_mapping = {

    "Dưới 30 thông báo/ngày": 0,
    "30-80 thông báo/ngày": 1,
    "80-150 thông báo/ngày": 2,
    "Trên 150 thông báo/ngày": 3

}

cleaned_dataset["notification_count"] = (
    cleaned_dataset["notification_count"]
    .replace(notification_mapping)
)

print("Đã mã hóa notification_count.")

Đã mã hóa notification_count.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\526251388.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(notification_mapping)


In [19]:
# Mã hóa số lần mở khóa điện thoại

unlock_mapping = {

    "Dưới 30 lần/ngày": 0,
    "30-60 lần/ngày": 1,
    "60-100 lần/ngày": 2,
    "Trên 100 lần/ngày": 3
}


cleaned_dataset["smartphone_unlocks"] = (
    cleaned_dataset["smartphone_unlocks"]
    .replace(unlock_mapping)
)


print("Đã mã hóa smartphone_unlocks.")

Đã mã hóa smartphone_unlocks.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\313375368.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(unlock_mapping)


In [20]:
# Mã hóa tần suất chuyển đổi ứng dụng

app_switch_mapping = {

    "Dưới 20 lần": 0,
    "20-50 lần": 1,
    "50-100 lần": 2,
    "Trên 100 lần": 3
}


cleaned_dataset["app_switch_frequency"] = (
    cleaned_dataset["app_switch_frequency"]
    .replace(app_switch_mapping)
)


print("Đã mã hóa app_switch_frequency.")

Đã mã hóa app_switch_frequency.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\42817520.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(app_switch_mapping)


Các biến Digital Exposure đã được chuyển đổi sang dạng ordinal numeric.

Phương pháp mã hóa này giữ được thứ tự mức độ sử dụng thiết bị:

- Giá trị thấp biểu thị mức độ tiếp xúc thấp.
- Giá trị cao biểu thị mức độ tiếp xúc cao.


Việc sử dụng ordinal encoding phù hợp với đặc điểm dữ liệu khảo sát và tránh đưa ra giả định không cần thiết về khoảng cách giữa các nhóm trả lời.

## 2.3.2 Cognitive Performance Encoding

Mã hóa các biến phản ánh khả năng tập trung và hiệu suất học tập.

- distraction_frequency
- focus_sessions
- deep_work_hours
- task_completion_rate

In [21]:
# Mã hóa biến distraction_frequency theo mức độ phân tâm tăng dần

distraction_mapping = {

    "0-2 lần (mình kiểm soát được khá tốt)": 0,
    "3-5 lần": 1,
    "6-10 lần": 2,
    "Trên 10 lần (mình gần như bị ngắt liên tục)": 3

}

cleaned_dataset["distraction_frequency"] = (
    cleaned_dataset["distraction_frequency"]
    .replace(distraction_mapping)
)

print("Đã mã hóa distraction_frequency.")

Đã mã hóa distraction_frequency.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\1386552233.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(distraction_mapping)


In [22]:
# Mã hóa số lần có phiên học/làm việc tập trung liên tục

focus_sessions_mapping = {

    "Không có lần nào": 0,
    "1-2 lần": 1,
    "3-4 lần": 2,
    "5 lần trở lên": 3

}

cleaned_dataset["focus_sessions"] = (
    cleaned_dataset["focus_sessions"]
    .replace(focus_sessions_mapping)
)

print("Đã mã hóa focus_sessions.")

Đã mã hóa focus_sessions.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\2782584151.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(focus_sessions_mapping)


In [23]:
# Mã hóa biến deep_work_hours

deep_work_mapping = {
    "Dưới 1 giờ": 0,
    "1-2 giờ": 1,
    "2-3 giờ": 2,
    "Trên 3 giờ": 3
}

cleaned_dataset["deep_work_hours"] = (
    cleaned_dataset["deep_work_hours"]
    .replace(deep_work_mapping)
    .astype("int64")
)

print("Đã mã hóa deep_work_hours.")

Đã mã hóa deep_work_hours.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\1245718765.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(deep_work_mapping)


In [24]:
# Mã hóa tỷ lệ hoàn thành công việc

task_completion_mapping = {

    "Dưới 25% - hầu như không làm được gì": 0,
    "25-50%": 1,
    "50-75%": 2,
    "Trên 75% - hoàn thành tốt": 3

}

cleaned_dataset["task_completion_rate"] = (
    cleaned_dataset["task_completion_rate"]
    .replace(task_completion_mapping)
)

print("Đã mã hóa task_completion_rate.")

Đã mã hóa task_completion_rate.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\245122030.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(task_completion_mapping)


Các biến tần suất đã được chuyển đổi sang dạng ordinal encoding.

Cách tiếp cận này giúp:

- Bảo toàn thứ tự mức độ của câu trả lời khảo sát.
- Giữ nguyên ý nghĩa định hướng của biến.
- Không tạo thêm đặc trưng mới trong giai đoạn Cleaning & Preprocessing.

Không thực hiện chuẩn hóa thang đo hoặc biến đổi phân phối trong bước này.

## 2.3.3 Sleep & Recovery Encoding

Chuẩn hóa các biến thuộc nhóm Sleep & Recovery nhằm đưa dữ liệu về dạng phù hợp cho các bước phân tích tiếp theo.

Nhóm biến Sleep & Recovery gồm:

- Thời lượng ngủ mỗi ngày.
- Chất lượng giấc ngủ.

Trong đó:

- `sleep_hours` là biến dạng khoảng giá trị nên cần thực hiện ordinal encoding.
- `sleep_quality` là biến Likert Scale từ 1 đến 10 nên chỉ kiểm tra và giữ nguyên giá trị.

In [25]:
# Mã hóa thời lượng ngủ trung bình mỗi đêm

sleep_hours_mapping = {

    "Dưới 5 tiếng": 0,
    "5-6 tiếng": 1,
    "6-7 tiếng": 2,
    "7-8 tiếng": 3,
    "Trên 8 tiếng": 4

}

cleaned_dataset["sleep_hours"] = (
    cleaned_dataset["sleep_hours"]
    .replace(sleep_hours_mapping)
)

print("Đã mã hóa sleep_hours.")

Đã mã hóa sleep_hours.


C:\Users\ASPIRE 7\AppData\Local\Temp\ipykernel_8848\2531274047.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(sleep_hours_mapping)


In [26]:
# Kiểm tra kiểu dữ liệu của sleep_quality

cleaned_dataset["sleep_quality"] = (
    pd.to_numeric(
        cleaned_dataset["sleep_quality"],
        errors="coerce"
    )
)


print("Đã kiểm tra sleep_quality.")

Đã kiểm tra sleep_quality.


In [27]:
# Kiểm tra khoảng giá trị của sleep_quality

sleep_quality_range = (
    cleaned_dataset["sleep_quality"]
    .describe()
)


sleep_quality_range

count    590.000000
mean       5.116949
std        1.943572
min        1.000000
25%        4.000000
50%        5.000000
75%        6.000000
max       10.000000
Name: sleep_quality, dtype: float64

Các biến thuộc nhóm Sleep & Recovery đã được xử lý phù hợp với đặc điểm dữ liệu khảo sát.

Cụ thể:

- `sleep_hours` được chuyển đổi sang dạng ordinal numeric nhằm phản ánh mức độ thời lượng ngủ.
- `sleep_quality` được giữ nguyên thang điểm Likert 1-10 nhằm bảo toàn ý nghĩa đánh giá chủ quan của người tham gia.

## 2.3.4 Burnout Symptoms Scale Validation

Kiểm tra và chuẩn hóa nhóm biến phản ánh triệu chứng Digital Burnout trong bộ dữ liệu khảo sát.

Nhóm biến Burnout Symptoms gồm các câu hỏi thuộc thang đo Digital Burnout Scale:

- Emotional Exhaustion.
- Stress Level.
- Mental Fatigue.


Các biến này sử dụng thang đo Likert 1-5:

- 1: Hoàn toàn không đồng ý.
- 2: Không đồng ý.
- 3: Không chắc / Tùy lúc.
- 4: Đồng ý.
- 5: Hoàn toàn đồng ý.


In [28]:
# Chuyển đổi các biến Burnout Scale sang dạng numeric

burnout_scale_columns = [

    "emotional_exhaustion",
    "stress_level",
    "mental_fatigue"

]

for column in burnout_scale_columns:

    cleaned_dataset[column] = (
        pd.to_numeric(
            cleaned_dataset[column],
            errors="coerce"
        )
    )


print("Đã kiểm tra các biến Burnout Symptoms.")

Đã kiểm tra các biến Burnout Symptoms.


In [29]:
# Kiểm tra khoảng giá trị của các biến Burnout Symptoms

burnout_scale_summary = (
    cleaned_dataset[burnout_scale_columns]
    .describe()
)


burnout_scale_summary

,emotional_exhaustion,stress_level,mental_fatigue
count,590.000000,590.000000,590.000000
mean,3.642373,3.988136,3.652542
std,1.008908,1.087759,1.102375
min,1.000000,1.000000,1.000000
25%,3.000000,3.250000,3.000000
50%,4.000000,4.000000,4.000000
75%,4.000000,5.000000,5.000000
max,5.000000,5.000000,5.000000


In [30]:
# Kiểm tra các giá trị ngoài phạm vi thang đo Burnout

invalid_burnout_values = {}

for column in burnout_scale_columns:

    invalid_values = (
        cleaned_dataset[column]
        [
            (cleaned_dataset[column] < 1) |
            (cleaned_dataset[column] > 5)
        ]
        .unique()
    )

    invalid_burnout_values[column] = invalid_values

invalid_burnout_values

{'emotional_exhaustion': array([], dtype=int64),
 'stress_level': array([], dtype=int64),
 'mental_fatigue': array([], dtype=int64)}

Các biến thuộc nhóm Burnout Symptoms đã được kiểm tra và chuẩn hóa về dạng numeric.

Kết quả kiểm tra nhằm đảm bảo:

- Các biến giữ đúng thang đo Likert 1-5.
- Không tồn tại giá trị ngoài phạm vi khảo sát.
- Không làm thay đổi ý nghĩa ban đầu của câu trả lời.


Các biến:

- `emotional_exhaustion`
- `stress_level`
- `mental_fatigue`

sẽ được sử dụng trong các bước phân tích Digital Burnout ở các notebook tiếp theo.

## 2.3.5 Final Encoding Validation

Kiểm tra tổng thể kết quả sau quá trình mã hóa các biến khảo sát nhằm đảm bảo dữ liệu đã được chuyển đổi đúng định dạng.

Phần này thực hiện kiểm tra:

- Các biến ordinal đã được chuyển sang dạng numeric.
- Các biến Likert Scale giữ đúng kiểu dữ liệu.
- Các biến categorical không bị thay đổi ngoài ý muốn.
- Phát hiện các giá trị text còn sót trong nhóm biến cần xử lý.

In [31]:
# Kiểm tra kiểu dữ liệu của toàn bộ dataset sau encoding

data_type_summary = (
    cleaned_dataset
    .dtypes
    .to_frame()
)

data_type_summary.columns = [
    "data_type"
]

data_type_summary

,data_type
gender,object
birth_year,object
education_stage,object
work_mode,object
device_usage_type,object
daily_screen_time,int64
social_media_hours,int64
doomscrolling_duration,int64
late_night_device_usage,int64
notification_count,int64


Bảng trên thể hiện kiểu dữ liệu của toàn bộ biến sau quá trình encoding.

Các biến định lượng và ordinal cần có kiểu dữ liệu numeric:

- int64
- float64

Các biến mô tả bối cảnh có thể tiếp tục giữ dạng object:

- gender
- birth_year
- education_stage
- work_mode
- device_usage_type

Các biến categorical này sẽ được xử lý phù hợp trong giai đoạn chuẩn bị dữ liệu cho mô hình Machine Learning.

## 2.3.5.2 Ordinal Encoding Validation

Đảm bảo các biến ordinal đã được chuyển đổi hoàn toàn sang dạng numeric.

Kiểm tra các biến:

- Digital Exposure.
- Cognitive Performance.
- Sleep Hours.

In [32]:
# Danh sách các biến ordinal đã encoding

ordinal_columns = [

    "daily_screen_time",
    "social_media_hours",
    "doomscrolling_duration",
    "late_night_device_usage",
    "notification_count",
    "smartphone_unlocks",
    "app_switch_frequency",

    "distraction_frequency",
    "focus_sessions",
    "deep_work_hours",
    "task_completion_rate",

    "sleep_hours"

]

# Kiểm tra datatype của các biến ordinal

ordinal_dtype_check = (
    cleaned_dataset[ordinal_columns]
    .dtypes
    .to_frame()
)


ordinal_dtype_check.columns = [
    "data_type"
]

ordinal_dtype_check

,data_type
daily_screen_time,int64
social_media_hours,int64
doomscrolling_duration,int64
late_night_device_usage,int64
notification_count,int64
smartphone_unlocks,int64
app_switch_frequency,int64
distraction_frequency,int64
focus_sessions,int64
deep_work_hours,int64


In [33]:
# Kiểm tra các giá trị không phải numeric trong nhóm ordinal

remaining_text_values = {}

for column in ordinal_columns:

    non_numeric_values = (
        cleaned_dataset[column]
        .apply(
            lambda x: isinstance(x, str)
        )
        .sum()
    )

    remaining_text_values[column] = non_numeric_values

remaining_text_values

{'daily_screen_time': np.int64(0),
 'social_media_hours': np.int64(0),
 'doomscrolling_duration': np.int64(0),
 'late_night_device_usage': np.int64(0),
 'notification_count': np.int64(0),
 'smartphone_unlocks': np.int64(0),
 'app_switch_frequency': np.int64(0),
 'distraction_frequency': np.int64(0),
 'focus_sessions': np.int64(0),
 'deep_work_hours': np.int64(0),
 'task_completion_rate': np.int64(0),
 'sleep_hours': np.int64(0)}

## 2.3.5.4 Categorical Variables Validation

Kiểm tra các biến categorical được giữ nguyên sau quá trình preprocessing.

Các biến categorical bao gồm:

- gender
- birth_year
- education_stage
- work_mode
- device_usage_type

Các biến này không thực hiện encoding trong Notebook 02.

In [34]:
# Kiểm tra số lượng giá trị duy nhất của biến categorical

categorical_columns = [

    "gender",
    "birth_year",
    "education_stage",
    "work_mode",
    "device_usage_type"

]

categorical_summary = (
    cleaned_dataset[categorical_columns]
    .nunique()
    .to_frame()
)

categorical_summary.columns = [
    "unique_values"
]

categorical_summary

,unique_values
gender,2
birth_year,5
education_stage,7
work_mode,4
device_usage_type,4


# 2.4 Data Consistency Validation

Đánh giá tính nhất quán của bộ dữ liệu sau quá trình Cleaning và Encoding.

Các kiểm tra được thực hiện gồm:

- Kiểm tra Missing Values phát sinh sau preprocessing.
- Kiểm tra Duplicate Records.
- Kiểm tra kiểu dữ liệu.
- Kiểm tra các giá trị không hợp lệ.

## 2.4.1 Missing Values Validation

Kiểm tra các giá trị thiếu còn tồn tại sau quá trình làm sạch và mã hóa.

Xác định:

- Số lượng missing values.
- Tỷ lệ missing values.

Nếu xuất hiện missing values:

- Kiểm tra nguyên nhân.
- Xác định có phải missing thực tế hay lỗi preprocessing.

In [35]:
# Kiểm tra missing values sau preprocessing

missing_summary = (
    cleaned_dataset
    .isnull()
    .sum()
    .to_frame()
)

missing_summary.columns = [
    "missing_count"
]

# Tính tỷ lệ missing

missing_summary["missing_rate"] = (
    missing_summary["missing_count"]
    /
    len(cleaned_dataset)
)

missing_summary.sort_values(
    by="missing_count",
    ascending=False
)

,missing_count,missing_rate
gender,0,0.0
birth_year,0,0.0
education_stage,0,0.0
work_mode,0,0.0
device_usage_type,0,0.0
daily_screen_time,0,0.0
social_media_hours,0,0.0
doomscrolling_duration,0,0.0
late_night_device_usage,0,0.0
notification_count,0,0.0


## 2.4.2 Duplicate Records Validation

Kiểm tra sự tồn tại của các bản ghi bị trùng lặp trong bộ dữ liệu sau quá trình làm sạch.

Thực hiện:

- Xác định số lượng bản ghi trùng lặp.
- Tính tỷ lệ bản ghi trùng trên toàn bộ dữ liệu.

In [36]:
# Kiểm tra số lượng bản ghi trùng lặp

duplicate_records = (
    cleaned_dataset
    .duplicated()
    .sum()
)

duplicate_rate = (
    duplicate_records
    /
    len(cleaned_dataset)
)

print("Số lượng bản ghi trùng:", duplicate_records)

print("Tỷ lệ bản ghi trùng:", duplicate_rate)

Số lượng bản ghi trùng: 24
Tỷ lệ bản ghi trùng: 0.04067796610169491


Kết quả kiểm tra cho thấy bộ dữ liệu tồn tại 24 bản ghi trùng lặp, chiếm khoảng 4.07% tổng số quan sát.

Các bản ghi trùng lặp được giữ nguyên trong bước Cleaning hiện tại vì:

- Dataset khảo sát không chứa mã định danh người tham gia.
- Chưa có đủ thông tin để xác định đây là các bản ghi không hợp lệ.
- Việc loại bỏ duplicate cần được xem xét dựa trên mục tiêu nghiên cứu và quy trình thu thập dữ liệu.

Do đó, các bản ghi trùng chỉ được ghi nhận để phục vụ đánh giá chất lượng dữ liệu.

Không thực hiện xóa duplicate trong bước này.

## 2.4.3 Data Type Validation

Kiểm tra và xác nhận kiểu dữ liệu của các biến sau quá trình Cleaning và Encoding.

Việc kiểm tra datatype nhằm đảm bảo:

- Các biến định lượng và biến ordinal đã được chuyển đổi sang dạng numeric phù hợp.
- Các biến phân loại vẫn giữ được dạng categorical.
- Không phát sinh lỗi kiểu dữ liệu trong quá trình xử lý.

In [37]:
# Kiểm tra kiểu dữ liệu sau preprocessing

data_type_summary = (
    cleaned_dataset
    .dtypes
    .to_frame()
)

data_type_summary.columns = [
    "data_type"
]

data_type_summary

,data_type
gender,object
birth_year,object
education_stage,object
work_mode,object
device_usage_type,object
daily_screen_time,int64
social_media_hours,int64
doomscrolling_duration,int64
late_night_device_usage,int64
notification_count,int64


## 2.4.4 Invalid Value Validation

Kiểm tra tính hợp lệ của các giá trị sau quá trình Cleaning và Encoding.

Mục tiêu của bước này là đảm bảo các biến được mã hóa vẫn nằm trong phạm vi giá trị được thiết kế từ bảng khảo sát ban đầu.

Thực hiện kiểm tra:

- Các biến sử dụng thang đo Likert 1-10.
- Các biến thuộc Digital Burnout Scale 1-5.
- Các biến ordinal sau encoding.
- Các giá trị ngoài phạm vi cho phép.

In [38]:
# Kiểm tra phạm vi giá trị của các biến Likert Scale

scale_validation = {

    "concentration_score": (
        cleaned_dataset["concentration_score"]
        .between(1, 10)
        .all()
    ),

    "sleep_quality": (
        cleaned_dataset["sleep_quality"]
        .between(1, 10)
        .all()
    ),

    "motivation_level": (
        cleaned_dataset["motivation_level"]
        .between(1, 10)
        .all()
    ),

    "emotional_exhaustion": (
        cleaned_dataset["emotional_exhaustion"]
        .between(1, 5)
        .all()
    ),

    "stress_level": (
        cleaned_dataset["stress_level"]
        .between(1, 5)
        .all()
    ),

    "mental_fatigue": (
        cleaned_dataset["mental_fatigue"]
        .between(1, 5)
        .all()
    )

}

scale_validation

{'concentration_score': np.True_,
 'sleep_quality': np.True_,
 'motivation_level': np.True_,
 'emotional_exhaustion': np.True_,
 'stress_level': np.True_,
 'mental_fatigue': np.True_}

# 3. Data Validation Summary

Tổng hợp và đánh giá lại toàn bộ quá trình Cleaning & Preprocessing trước khi xuất dataset sạch.

Phần này thực hiện:

- So sánh trạng thái dữ liệu trước và sau Cleaning.
- Tổng hợp các thay đổi đã thực hiện.
- Xác nhận dataset đáp ứng yêu cầu đầu vào cho Notebook 03 - Exploratory Data Analysis.

Các tiêu chí đánh giá:

- Số lượng bản ghi.
- Số lượng biến.
- Missing Values.
- Duplicate Records.
- Kiểu dữ liệu.

In [39]:
# Tổng hợp thông tin trước và sau cleaning

validation_summary = pd.DataFrame({

    "before_cleaning": [
        raw_dataset.shape[0],
        raw_dataset.shape[1],
        raw_dataset.isnull().sum().sum(),
        raw_dataset.duplicated().sum()
    ],

    "after_cleaning": [
        cleaned_dataset.shape[0],
        cleaned_dataset.shape[1],
        cleaned_dataset.isnull().sum().sum(),
        cleaned_dataset.duplicated().sum()
    ]

}, index=[
    "number_of_records",
    "number_of_features",
    "missing_values",
    "duplicate_records"
])

validation_summary

,before_cleaning,after_cleaning
number_of_records,641,590
number_of_features,25,23
missing_values,0,0
duplicate_records,24,24


In [40]:
# Tạo bảng tổng hợp quá trình cleaning

cleaning_summary = pd.DataFrame({

    "processing_step": [

        "Column standardization",
        "Attention check removal",
        "Text value standardization",
        "Ordinal encoding",
        "Missing values validation",
        "Duplicate records validation",
        "Data type validation",
        "Invalid value validation"

    ],

    "status": [

        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed"

    ]

})

cleaning_summary

,processing_step,status
0,Column standardization,Completed
1,Attention check removal,Completed
2,Text value standardization,Completed
3,Ordinal encoding,Completed
4,Missing values validation,Completed
5,Duplicate records validation,Completed
6,Data type validation,Completed
7,Invalid value validation,Completed


# 4. Save Clean Dataset

Xuất bộ dữ liệu đã được làm sạch và tiền xử lý để sử dụng cho Notebook 03 - Exploratory Data Analysis.

Thực hiện:

- Kiểm tra đường dẫn lưu dữ liệu.
- Tạo thư mục output nếu chưa tồn tại.
- Xuất dataset sau Cleaning dưới định dạng CSV.

In [41]:
# Thiết lập đường dẫn output cho dataset đã làm sạch

import os


output_dir = "../../data/processed/vietnam_dataset"

output_path = os.path.join(
    output_dir,
    "vn_digital_burnout_cleaned.csv"
)


# Tạo thư mục nếu chưa tồn tại

os.makedirs(
    output_dir,
    exist_ok=True
)


print("Đã thiết lập đường dẫn lưu dữ liệu.")

Đã thiết lập đường dẫn lưu dữ liệu.


In [42]:
# Xuất dataset đã làm sạch

cleaned_dataset.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)


print("Đã xuất dataset sau cleaning.")

Đã xuất dataset sau cleaning.


In [43]:
# Kiểm tra file output sau khi export

if os.path.exists(output_path):

    exported_dataset = pd.read_csv(
        output_path
    )

    print("File dữ liệu đã được lưu thành công.")
    print("Số dòng:", exported_dataset.shape[0])
    print("Số cột:", exported_dataset.shape[1])

else:

    print("Không tìm thấy file dữ liệu.")

File dữ liệu đã được lưu thành công.
Số dòng: 590
Số cột: 23
